<a href="https://colab.research.google.com/github/Yurnero2706/DataAlgo-UT/blob/main/DataAlgo2026_R09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ構造とアルゴリズム 第9週課題 (2026/06/17 ver.A)

---


★下記３要素を**必ず直接書き換えてから**提出すること．

* 学籍番号： 202318032
* 氏名： Nguyen Cong Nguyen
* Colabアカウント： ncnguyencva@gmail.com

**学生同士で教えた・教わった・グループワークをした場合，必ず下記の当該行を直接書き換えて記述すること．**
申告があった場合は，論述などで内容が似ていたり，プログラムが似ていても減点はしない．（コピペレベルの同一文などは処罰対象）
**教えた側は加点対象となる．**

**[教えた側]**
教えた相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；教えた相手の内容が浅くなってないか確認すること）

**[教わった側]**
教わった相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；必ず教えた側に自分の名前を書いてもらうこと）

**[グループワーク]**
一緒に行った相手：＜氏名＞　＜学籍番号＞
（何名いてもよいが，必ずお互いの名前を全員記すこと）

---
# 必須課題9A ナップサック問題の計算量に関する考察

DataAlgo_UT(014)_KnapSack_BB.ipynb の Knsapsack-bf_Jプログラム, Knapsack-bt_Jプログラム, およびKnapsack-bb_Jプログラムそれぞれについて，時間計算量と空間計算量について考察せよ．特に，時間計算量については，幾つかのプログラムにおいては，ビッグオー表現では計算量が微妙な結果になってしまう状況が考えられる．その場合に，授業での解説を踏まえて解の正当性を説明せよ．



Let $N$ be the number of objects.

**`Knapsack-bf_J` (brute force).**: It visits every node of the full binary tree of depth $N$, so $\Theta(2^N)$ calls of `pickobject()`.
- Time: $O(2^N)$
- Space: $O(N)$ (`stat[N]` + recursion stack).

**`Knapsack-bt_J` (backtrack).**: The program skips the "pick" branch when the new weight would exceed `wlimit`.
- Time: $O(2^N)$
- Space: $O(N)$.

**`Knapsack-bb_J` (branch-and-bound).** Same as backtrack, plus skip the "discard" branch when even the best remaining achievable value cannot beat `max.v`.
- Worst case: still $O(2^N)$.
- Space: $O(N)$ (one extra `int` per frame for `av`).

**The "subtle result".**

All three programs have the **same** big-O time $O(2^N)$, even though bb is dramatically faster in practice. Big-O captures *worst* case, and in the worst case the pruning may never fire.

**Why this is still valid.** The pruning is *safe* — backtrack only skips branches that violate the weight limit, branch-and-bound only skips branches whose best possible value cannot beat the current best. Both kinds of branch cannot improve the optimizattion, so they are correctly removed. So backtrack and branch-and-bound always return the same optimal answer as brute force; they are just faster in typical cases.

---
# 必須課題9B ナップサック問題を解くプログラムの計算量の実測

DataAlgo_UT(014)_KnapSack_BB.ipynb で示した Knsapsack-bf_Jプログラム, Knapsack-bt_Jプログラム, およびKnapsack-bb_Jプログラムのそれぞれについて，計算量を実際に測定せよ．pickobject()関数の呼び出し回数を数えるプログラムを作成し，その数で3プログラムの計算コストを比較すること．ナップサックの重み制限を変更して，その変化を表にまとめること．重み制限を変えて10回以上実験すること．（つまり下記の表で10行以上のデータが集まるようにすること)
結果を表にまとめ，考察もすること．

|制限|bf|bt|bb|
|---|--|--|--|
|100|511|337|17|
|125|511|438|15|
|150|511|489|18|
|175|511|507|12|
|200|511|511|9|
|300|511|511|9|

計算量の測定方法については，解候補を数える方法と今回の方法とで結果が異なる．
その違いの理由を説明し，二つの測定方法のの優劣について論じること．


I got the result as follow:
|制限|bf|bt|bb|
|---|--|--|--|
|100|511|337|17|
|125|511|438|15|
|150|511|489|18|
|175|511|507|12|
|200|511|511|9|
|300|511|511|9|

**Why this measure differs from "counting solution candidates".**

- "Candidates examined" = number of *leaves* of the recursion tree.
- `pickobject` calls = number of *all nodes* (internal + leaves).

So this measure is roughly **2× larger** than the leaf count, and also picks up the work done at pruned internal nodes (which is real CPU work that the leaf count ignores).

**Pros and cons.**

- *Counting leaves:* clean theoretical bound ($2^N$), implementation-independent, but hides the cost of pruning checks at internal nodes.
- *Counting `pickobject` calls:* closer to actual CPU work, fairly penalises pruning that visits-then-discards, but implementation-dependent (an iterative rewrite would give a different number).

---
# 必須課題9C 問題の還元にかかる計算量
0-1ナップサック問題(6.3.節参照)を最長経路問題に還元するのに必要な時間計算量と空間計算量を，導出過程を含めて示せ．0-1ナップサック問題の品物の数を正整数N，ナップサック重量制限を正整数WLとする．各品物の重量は正整数で表されるものとする．


- Vertices: $V = \{(w, i) \mid i = 0..N,\; w = 0..WL\}$. So $|V| = (N+1)(WL+1)$.
- Edges (three kinds): "don't pick" (vertical, weight 0), "filler at top row" (horizontal, weight 0), "pick" (up-right, weight $V_i$, only when $w + W_i \leq WL$). So $|E| \leq 2N(WL+1) + WL$.

Both counts are $O(N \cdot WL)$.

**Time complexity of the reduction.**

We enumerate every vertex and edge once with $O(1)$ work each.
$$\text{Time} = O(|V| + |E|) = O(N \cdot WL).$$

**Space complexity of the reduction.**

- Adjacency list: $O(|V| + |E|) = O(N \cdot WL)$.
- Adjacency matrix (what `longestpath_J` expects): $O(|V|^2) = O((N \cdot WL)^2)$.

 This is *pseudo-polynomial*: polynomial in the *value* of $WL$, but $WL$'s bit length $b$ satisfies $WL \leq 2^b$, so the reduction is exponential in $b$. That is why this construction does not contradict 0-1 knapsack being NP-hard — it is only efficient when $WL$ is small.

---
---

# 発展課題9X 最小全域木を求めるプリムのアルゴリズム
本授業において，最小全域木が必要になる状況がどのようなものか，githubなども参照して説明せよ．
次に，最小全域木を求めるPrimのアルゴリズムを解説し，最小経路問題の解法の一つであるダイクストラのアルゴリズムとの相似点について言及せよ．何かしらの出典を見て纏めてももちろんよいが，その出典をURL等で明確に示すこと．
ChatGPT等に相談してもよいが，その場合でも，そのアルゴリズムの出典を明確に示すとともに，その内容が確認できるURL等を提示すること．  
（一般論として，ChatGPT利用時は『出典を正確に示すこと』．）  

【注意】  
出典を辿ってみて元資料を閲覧しても，ChatGPTが指摘している内容がどこに記載されているかすぐには分からない場合が（結構）ある．この確認をサボると，ChatGPT自身の性能が何かの拍子に劣化したときに，まったく確認ができてないままとなってしまう．そのため，自分の目と頭で出典とChatGPTの示す内容が一致するか確認すること．


（議論はここに記述）

---
# 課題提出法

筑波大学工学システム学類３年生向け．
FG24711 / FG34711．

必須課題は全て実施すること．
発展課題はしなくともよいが，A+取得には発展課題を（全課題提出を通して）1つ以上実施していることが必要条件である．

該当するmanabaに課題提出のエントリを設けるので，そこに本**Colab notebookを提出**すること．

----
# 出典

筑波大学工学システム学類
データ構造とアルゴリズム
担当：亀田能成


---
2026/06/17 ver.A  
